In [23]:
import streamlit as st
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [24]:
st.set_page_config(page_title="FallUp Scraper", layout="centered")

# --- Keyword mapping ---
INDUSTRY_KEYWORDS = {
    "Artificial Intelligence": ["ai", "machine learning", "deep learning", "neural network"],
    "Information Technology": ["cloud", "software", "infrastructure", "devops", "it services"],
    "Human Resources": ["recruitment", "talent", "hr software", "onboarding", "employee"],
    "E-commerce": ["shop", "cart", "checkout", "ecommerce", "store", "buy"],
    "Finance": ["investment", "banking", "fintech", "insurance", "financial"],
    "Healthcare": ["healthcare", "clinic", "hospital", "medical", "pharma"]
}

# --- Helper Functions ---
def normalize_url(domain):
    if not domain.startswith("http"):
        return "http://" + domain
    return domain

def classify_industry(text):
    if not isinstance(text, str):
        return "Unknown"
    text = text.lower()
    for industry, keywords in INDUSTRY_KEYWORDS.items():
        if any(keyword in text for keyword in keywords):
            return industry
    return "Other"

def scrape_website(domain):
    url = normalize_url(domain)
    try:
        response = requests.get(url, timeout=8)
        soup = BeautifulSoup(response.content, "html.parser")
        title = soup.title.string if soup.title else "No title"
        text = soup.get_text()
        industry = classify_industry(text)

        links = [a.get("href") for a in soup.find_all("a", href=True)]
        linkedin = next((l for l in links if "linkedin.com" in l), None)
        twitter = next((l for l in links if "twitter.com" in l), None)
        facebook = next((l for l in links if "facebook.com" in l), None)

        return {
            "Domain": domain,
            "Title": title,
            "Predicted Industry": industry,
            "LinkedIn": linkedin,
            "Twitter": twitter,
            "Facebook": facebook
        }
    except Exception as e:
        return {"Domain": domain, "Error": str(e)}

# --- Streamlit UI ---
st.title("FallUp Domain Scraper")
st.markdown("Enter a domain to extract industry & social info:")

scraped_data = []
domain_input = st.text_input("Enter Domain (e.g., example.com)")

if st.button("Scrape Now") and domain_input:
    with st.spinner("Scraping the website..."):
        result = scrape_website(domain_input)
    scraped_data.append(result)
    st.success("Scraping complete!")
    for key, value in result.items():
        st.markdown(f"**{key}:** {value}")

    # Convert to DataFrame for download
    df_result = pd.DataFrame([result])
    csv = df_result.to_csv(index=False).encode('utf-8')
    st.download_button(
        label="📥 Download Result as CSV",
        data=csv,
        file_name=f"scraped_{domain_input.replace('.', '_')}.csv",
        mime="text/csv"
    )

In [35]:
import pandas as pd

# Lead scoring function returning float (0 to ~5)
def compute_star_rating(row):
    score = 0

    # Employee count
    emp = row.get("number_of_employees", 0)
    if emp >= 5000:
        score += 1
    elif emp >= 1000:
        score += 0.75
    elif emp >= 100:
        score += 0.5
    else:
        score += 0.25

    # Share price growth
    try:
        growth = float(row.get("share_price_current", 0)) - float(row.get("share_price_5y_ago", 0))
        if growth > 200:
            score += 1
        elif growth > 100:
            score += 0.75
        elif growth > 0:
            score += 0.5
        else:
            score += 0.25
    except:
        score += 0.25

    # Email domain
    email = str(row.get("contact_email", ""))
    domain = email.split("@")[-1] if "@" in email else ""
    generic_domains = ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com"]
    score += 1 if domain not in generic_domains else 0.25

    # Leadership title
    title = str(row.get("contact_title", "")).lower()
    if any(keyword in title for keyword in ["chief", "ceo", "founder", "president", "vp", "head", "director"]):
        score += 1
    else:
        score += 0.25

    return round(score, 2)  # Float star rating

# Load file
file_path = r"C:\Users\hp\AppData\Roaming\Microsoft\Windows\Start Menu\demo_company_dataset.csv"
df = pd.read_csv(file_path)

# Compute float star rating
df["star_rating"] = df.apply(compute_star_rating, axis=1)

# Print result
print("\n✅ Leads scored successfully!\n")
print(df[["company_name", "star_rating"]])



✅ Leads scored successfully!

                     company_name  star_rating
0              Cunningham-Ramirez         2.50
1                        Chen LLC         2.00
2                     Porter-Frey         2.25
3                      Campos-Kim         2.50
4     Thompson, Harris and Turner         2.75
..                            ...          ...
995                  Copeland Ltd         2.75
996  Watson, Hubbard and Martinez         3.00
997                 Garrett-Evans         2.50
998        Kidd, Brown and Martin         2.25
999                     Patel Ltd         2.50

[1000 rows x 2 columns]
